# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [ ]:
print('hello123')

hello123


In [ ]:
# # RTX 4060(Windows) 기준 권장 설치: CUDA 12.1 stable wheel
# # Python 3.13 환경이면 stable wheel이 없을 수 있어 nightly로 fallback 하세요.
# !pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# # fallback (필요시): !pip install --pre -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 10.7 MB/s eta 0:03:49
     ---------------------------------------- 0.0/2.4 GB 11.6 MB/s eta 0:03:31
     ---------------------------------------- 0.0/2.4 GB 11.6 MB/s eta 0:03:31
     ---------------------------------------- 0.0/2.4 GB 11.7 MB/s eta 0:03:29
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:27
     ---------------------------------------- 0.0/2.4 GB 11.7 MB/s eta 0:03:28
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:27
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:26
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:26
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:26
     ---------------------------------------- 0.0/2.4 GB 11.8 MB/s eta 0:03:25
 

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

2.5.1+cu121
True
NVIDIA GeForce RTX 4060


In [ ]:
# 압축 해제
!unzip "/content/test.zip" -d "/content/"
!unzip "/content/train.zip" -d "/content/"

In [ ]:
# # Qwen3-VL + Windows 환경 권장 버전(모듈 포함)
# !pip -q install -U "transformers==4.57.1" "accelerate==1.1.1" \
#   "peft==0.14.0" "bitsandbytes>=0.45.4" "huggingface_hub==0.36.0" \
#   datasets pillow pandas scipy
# # 설치 후에는 반드시 커널 재시작



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# HF Hub 인증 (권장): 속도/요청 제한 개선
import os
from huggingface_hub import login

# 1) 환경변수 HF_TOKEN 우선 사용
# 2) 없으면 로컬 .env 파일에서 HF_TOKEN=... 형태를 읽어 사용
if not os.getenv("HF_TOKEN") and os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("HF_TOKEN="):
                os.environ["HF_TOKEN"] = line.split("=", 1)[1].strip().strip('"').strip("'")
                break

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("HF Hub login success")
else:
    print("HF_TOKEN not found. (비로그인 상태로 진행 가능)")

HF_TOKEN not found. (비로그인 상태로 진행 가능)


c:\Users\kimgm\Desktop\ssafy_ai_competition\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# hf_xet은 선택 설치(필수 아님)
!pip -q install -U hf_xet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Colab 데이터 준비: 압축 업로드/해제 후 사용
# 아래 구조가 되도록 준비하세요.
# /content/train.csv, /content/test.csv, /content/train/, /content/test/
%cd /content
import os
print('cwd:', os.getcwd())
print('train.csv exists:', os.path.exists('train.csv'))
print('test.csv exists:', os.path.exists('test.csv'))


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 속도 최적화 (Ampere+ GPU에서 효과)
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# OOM 방지: 학습은 저해상도, 추론은 고해상도 사용
TRAIN_TARGET_WIDTH = 512
TRAIN_TARGET_HEIGHT = 384
TRAIN_TARGET_PIXELS = TRAIN_TARGET_WIDTH * TRAIN_TARGET_HEIGHT

INFER_TARGET_WIDTH = 1024
INFER_TARGET_HEIGHT = 768
INFER_TARGET_PIXELS = INFER_TARGET_WIDTH * INFER_TARGET_HEIGHT

MAX_NEW_TOKENS = 2
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 학습데이터 2000개만 추출 (데이터가 더 적으면 전체 사용)
train_df = train_df.sample(n=min(2000, len(train_df)), random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 프로세서 로드 전 패치:
# 일부 버전 조합에서 additional_chat_templates 404를 예외로 처리하지 못해 중단되는 문제를 우회
import transformers.utils.hub as tf_hub
import transformers.tokenization_utils_base as tub

# huggingface_hub 버전에 따라 예외 클래스명이 달라서 호환 처리
try:
    from huggingface_hub.errors import RemoteEntryNotFoundError as _HFNotFoundError
except Exception:
    try:
        from huggingface_hub.errors import EntryNotFoundError as _HFNotFoundError
    except Exception:
        _HFNotFoundError = Exception

_orig_list_repo_templates = tf_hub.list_repo_templates


def _safe_list_repo_templates(*args, **kwargs):
    try:
        return _orig_list_repo_templates(*args, **kwargs)
    except _HFNotFoundError:
        return []


tf_hub.list_repo_templates = _safe_list_repo_templates
# tokenization_utils_base가 import 시점에 심볼을 잡고 있어 같이 패치
tub.list_repo_templates = _safe_list_repo_templates

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=TRAIN_TARGET_PIXELS,
    max_pixels=INFER_TARGET_PIXELS,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.09s/it]


trainable params: 16,515,072 || all params: 4,454,330,880 || trainable%: 0.3708


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a visual multiple-choice classifier for recycling items. "
    "Most questions ask item-category or material-type. "
    "Use image evidence first, then match options. "
    "Focus on dominant signals: material (plastic/metal/glass/paper/vinyl), "
    "item category (container/can/bottle/tray/cup), and visible parts (label/cap/body). "
    "If two options look similar, choose the one that best matches the main body material or object type. "
    "Return exactly one lowercase letter: a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        "다음은 재활용 분류 객관식 문제입니다.\n"
        "질문은 주로 '무엇/어떤' 유형의 품목 분류 또는 재질 판별입니다.\n"
        "이미지의 본체 재질과 물체 종류를 우선 판단하고, 선택지와 가장 정확히 매칭하세요.\n\n"
        f"질문: {question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "출력 규칙: 정답은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력."
    )

def resize_with_padding(img, target_w=TRAIN_TARGET_WIDTH, target_h=TRAIN_TARGET_HEIGHT, fill=(0, 0, 0)):
    # 종횡비를 유지하고 남는 영역은 패딩으로 채웁니다.
    w, h = img.size
    scale = min(target_w / max(w, 1), target_h / max(h, 1))
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = img.resize((new_w, new_h), Image.BICUBIC)

    canvas = Image.new("RGB", (target_w, target_h), fill)
    left = (target_w - new_w) // 2
    top = (target_h - new_h) // 2
    canvas.paste(resized, (left, top))
    return canvas

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        img = resize_with_padding(img, TRAIN_TARGET_WIDTH, TRAIN_TARGET_HEIGHT)

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)

# OOM 완화: gradient accumulation 증가 + cache 비활성화
GRAD_ACCUM = 8
if hasattr(model, "config"):
    model.config.use_cache = False

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

# 학습 루프
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen3_vl_4b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


Epoch 1 [train]:   0%|          | 0/1800 [00:00<?, ?batch/s]

Epoch 1 [train]:   0%|          | 2/1800 [01:15<18:54:46, 37.87s/batch]

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# Logit 기반 선택: 마지막 토큰 위치에서 a/b/c/d 확률 비교
def build_choice_token_ids(tokenizer):
    choice_token_ids = {}
    for ch in ["a", "b", "c", "d"]:
        cand_ids = set()
        for variant in [ch, " " + ch, "(" + ch + ")", ch + "."]:
            ids = tokenizer.encode(variant, add_special_tokens=False)
            if len(ids) >= 1:
                cand_ids.add(ids[-1])
        choice_token_ids[ch] = sorted(cand_ids)
    return choice_token_ids

def pick_choice_from_logits(last_logits, choice_token_ids):
    # 각 선택지에 대해 후보 토큰들의 최대 logit을 대표값으로 사용
    scores = {}
    for ch, ids in choice_token_ids.items():
        if len(ids) == 0:
            scores[ch] = -1e9
        else:
            idx = torch.tensor(ids, device=last_logits.device)
            scores[ch] = torch.max(last_logits.index_select(dim=-1, index=idx)).item()
    return max(scores, key=scores.get)

model.eval()
preds = []
choice_token_ids = build_choice_token_ids(processor.tokenizer)

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    img = resize_with_padding(img, INFER_TARGET_WIDTH, INFER_TARGET_HEIGHT)
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        last_logits = outputs.logits[0, -1, :]
        pred = pick_choice_from_logits(last_logits, choice_token_ids)

    preds.append(pred)

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")


Inference:   0%|          | 0/3887 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
c:\Users\SSAFY\anaconda3\envs\baseline4\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Inference:   8%|▊         | 305/3887 [01:40<19:35,  3.05sample/s]


KeyboardInterrupt: 

In [ ]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
이 사진에 보이는 음식 중 치킨 너겟과 함께 제공된 음식은 무엇인가요?
(a) 피클
(b) 감자튀김
(c) 케첩
(d) 셀러리 스틱과 소스

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
d
